# Experiment 13: Literature-Grounded Architecture Sweep (Phase B)

## Motivation
Following the identifiability audit (Experiment 12) which confirmed an information limit at the current SNR/PLD configuration, we pivot from generic sequence models to highly controlled, literature-supported architectures. 

## Experimental Arms
1. **Experiment A (Deep MLP):** The 2024 NMR Biomedicine paper evaluated standardized MLPs up to 9 hidden layers. We test 3, 5, 7, and 9 hidden layers (100 neurons each) to see if depth resolves the non-linear inverse problem better than the baseline 8-layer MLP.
2. **Experiment B (PLD-Aware Attention):** Rather than generic sequences, we construct explicit (PLD_time, Signal) pairs and feed them into an attention mechanism, allowing the model to dynamically weight PLD information content.
3. **Experiment C (Physics-Informed DNN - PINN):** We implement the $\lambda_{phys}$ reconstruction loss ($L = L_{param} + \lambda L_{phys}$) from Ishida et al. (2024), scaling $\lambda \in [0.0, 0.01, 0.1, 0.5]$ to regularize the degenerate parameter space.
4. **Experiment D (Sensitivity-Aware Loss):** Standard MSE heavily penalizes CBF (scale 20-90) while ignoring ATT (scale 0.5-3.0) relatively. We apply Z-score standardized MAE and weighted losses.


In [1]:
import sys, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd().parent if not (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(project_root / "src"))
from simulation import SimulationConfig, generate_dataset, paper_signal

out_root = project_root / "results" / "exp13_literature_architectures"
out_root.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = SimulationConfig(reference_cbf=50.0, reference_att_s=1.6)

# Standardize exactly as before
X_tr, Y_tr, _ = generate_dataset(100000, cfg, seed=42, snr=10.0)
X_va, Y_va, _ = generate_dataset(20000, cfg, seed=123, snr=10.0)
X_te, Y_te, _ = generate_dataset(20000, cfg, seed=999, snr=10.0)

X_m = X_tr.mean(0, keepdims=True); X_s = X_tr.std(0, keepdims=True) + 1e-8
CBF_m, CBF_s = Y_tr[:,0].mean(), Y_tr[:,0].std()
ATT_m, ATT_s = Y_tr[:,1].mean(), Y_tr[:,1].std()

def norm_X(x): return (x - X_m) / X_s
def norm_Y(y): return np.column_stack(((y[:,0]-CBF_m)/CBF_s, (y[:,1]-ATT_m)/ATT_s))

# Prepare PyTorch Datasets
def to_dl(X, Y, b_size=1024, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(norm_X(X)).float().to(device),
                                    torch.from_numpy(norm_Y(Y)).float().to(device)), 
                      batch_size=b_size, shuffle=shuffle)

train_loader = to_dl(X_tr, Y_tr, shuffle=True)
val_loader = to_dl(X_va, Y_va)
test_loader = to_dl(X_te, Y_te)


In [2]:
def torch_paper_signal_2x(cbf, att, plds, cfg):
    '''
    Differentiable Buxton signal model for PINN loss.
    NOTE: The simulator's noisy_signals output is essentially ~2x the pure clean signal due to mc + ml.
    We apply the 2.0 multiplier to align the PINN physics reconstruction scale with the input X.
    '''
    delta = att.unsqueeze(1)
    flow = (cbf / (6000.0 * cfg.lambda_blood)).unsqueeze(1)
    plds_t = torch.tensor(plds, device=cbf.device, dtype=torch.float32).unsqueeze(0)
    
    prefix = 2.0 * cfg.alpha * cfg.beta * cfg.t1_tissue_s * flow
    first = torch.exp(-torch.relu(plds_t - delta) / cfg.t1_tissue_s)
    second = torch.exp(-torch.relu(cfg.tau_s + plds_t - delta) / cfg.t1_tissue_s)
    clean = cfg.scale * prefix * torch.exp(-delta / cfg.t1_blood_s) * (first - second)
    return clean * 2.0


In [3]:
def train_model(model, name, plds, epochs=50, phys_lambda=0.0):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5)
    
    # We dynamically select 3 PLDs or 6 PLDs
    idx = [0,1,2,3,4,5] if len(plds)==6 else [0,2,3] # 1.525, 2.525, 3.025
    
    best_val = float('inf')
    for ep in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx_sub = bx[:, idx]
            opt.zero_grad()
            preds = model(bx_sub)
            
            loss_param = nn.L1Loss()(preds, by)
            
            loss_total = loss_param
            if phys_lambda > 0.0:
                # Denormalize predictions for physics simulator
                pred_cbf = (preds[:, 0] * CBF_s) + CBF_m
                pred_att = (preds[:, 1] * ATT_s) + ATT_m
                # Reconstruct signal
                pred_sig = torch_paper_signal_2x(pred_cbf, pred_att, plds, cfg)
                # Compare reconstructed signal to the original noisy input (denormalized)
                true_sig = (bx_sub * torch.tensor(X_s[:, idx], device=device)) + torch.tensor(X_m[:, idx], device=device)
                loss_phys = nn.L1Loss()(pred_sig, true_sig)
                loss_total = loss_param + phys_lambda * loss_phys
                
            loss_total.backward()
            opt.step()
            
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                val_loss += nn.L1Loss()(model(bx[:, idx]), by).item()
        
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), out_root / f"{name}.pt")
            
    # Evaluation
    model.load_state_dict(torch.load(out_root / f"{name}.pt"))
    model.eval()
    preds_all = []
    with torch.no_grad():
        for bx, _ in test_loader:
            preds_all.append(model(bx[:, idx]).cpu().numpy())
    preds_all = np.vstack(preds_all)
    
    # Denormalize and clip
    pred_cbf = np.clip((preds_all[:, 0] * CBF_s) + CBF_m, 0.0, 100.0)
    pred_att = np.clip((preds_all[:, 1] * ATT_s) + ATT_m, 0.5, 3.0)
    
    cbf_rmse = np.sqrt(np.mean((pred_cbf - Y_te[:, 0])**2))
    att_rmse = np.sqrt(np.mean((pred_att - Y_te[:, 1])**2))
    return cbf_rmse, att_rmse


In [4]:
class DeepMLP(nn.Module):
    def __init__(self, in_dim, num_layers=5):
        super().__init__()
        layers = [nn.Linear(in_dim, 100), nn.ELU()]
        for _ in range(num_layers-1):
            layers += [nn.Linear(100, 100), nn.ELU()]
        layers.append(nn.Linear(100, 2))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class PLDAwareNet(nn.Module):
    def __init__(self, in_dim, plds):
        super().__init__()
        self.plds = torch.tensor(plds, dtype=torch.float32)
        # Process (Signal, PLD) pairs
        self.feature_net = nn.Sequential(nn.Linear(2, 32), nn.ELU(), nn.Linear(32, 32), nn.ELU())
        self.attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
        self.regressor = nn.Sequential(nn.Linear(32, 64), nn.ELU(), nn.Linear(64, 2))
        
    def forward(self, x):
        batch = x.shape[0]
        seq_len = x.shape[1]
        p = self.plds.to(x.device).unsqueeze(0).expand(batch, -1).unsqueeze(2) # [B, S, 1]
        sig = x.unsqueeze(2) # [B, S, 1]
        pair = torch.cat([sig, p], dim=-1) # [B, S, 2]
        
        feats = self.feature_net(pair) # [B, S, 32]
        attn_out, _ = self.attention(feats, feats, feats) # [B, S, 32]
        
        # Global pool
        pooled = attn_out.mean(dim=1) # [B, 32]
        return self.regressor(pooled)

print("Architectures defined.")


Architectures defined.


In [5]:
results = []
plds_6 = cfg.plds_6_s
plds_3 = (1.525, 2.525, 3.025)

# A. Deep MLP Sweep
for layers in [3, 5, 7, 9]:
    print(f"Training DeepMLP (Layers={layers})...")
    c6, a6 = train_model(DeepMLP(6, layers), f"MLP_{layers}_6", plds_6)
    c3, a3 = train_model(DeepMLP(3, layers), f"MLP_{layers}_3", plds_3)
    results.append({"Model": f"DeepMLP-{layers}", "6-CBF": c6, "6-ATT": a6, "3-CBF": c3, "3-ATT": a3})

# B. PLD-Aware Net
print("Training PLD-Aware Attention...")
c6, a6 = train_model(PLDAwareNet(6, plds_6), "PLDAware_6", plds_6)
c3, a3 = train_model(PLDAwareNet(3, plds_3), "PLDAware_3", plds_3)
results.append({"Model": "PLDAware Attention", "6-CBF": c6, "6-ATT": a6, "3-CBF": c3, "3-ATT": a3})

# C. Physics-Informed (PINN) Sweep on standard 5-layer MLP
for lmb in [0.01, 0.1, 0.5]:
    print(f"Training PINN (Lambda={lmb})...")
    c6, a6 = train_model(DeepMLP(6, 5), f"PINN_{lmb}_6", plds_6, phys_lambda=lmb)
    c3, a3 = train_model(DeepMLP(3, 5), f"PINN_{lmb}_3", plds_3, phys_lambda=lmb)
    results.append({"Model": f"PINN ($\lambda={lmb}$)", "6-CBF": c6, "6-ATT": a6, "3-CBF": c3, "3-ATT": a3})

df = pd.DataFrame(results)
df["3PLD CBF Penalty (%)"] = ((df["3-CBF"] - df["6-CBF"]) / df["6-CBF"]) * 100
df["3PLD ATT Penalty (%)"] = ((df["3-ATT"] - df["6-ATT"]) / df["6-ATT"]) * 100
df.to_csv(out_root / "exp13_results.csv", index=False)
display(df)


Training DeepMLP (Layers=3)...


C:\Users\ANU THOMSON\AppData\Local\Temp\ipykernel_19668\3844712803.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(out_root / f"{name}.

Training DeepMLP (Layers=5)...


Training DeepMLP (Layers=7)...


Training DeepMLP (Layers=9)...


Training PLD-Aware Attention...


Training PINN (Lambda=0.01)...


Training PINN (Lambda=0.1)...


Training PINN (Lambda=0.5)...


,Model,6-CBF,6-ATT,3-CBF,3-ATT,3PLD CBF Penalty (%),3PLD ATT Penalty (%)
0,DeepMLP-3,4.397781,0.371989,4.936773,0.389905,12.255998,4.816235
1,DeepMLP-5,4.388685,0.372650,4.927575,0.390745,12.279073,4.855921
2,DeepMLP-7,4.400461,0.368443,4.947746,0.390526,12.437006,5.993551
3,DeepMLP-9,4.381320,0.369273,4.929697,0.389451,12.516253,5.464348
4,PLDAware Attention,4.488523,0.374370,5.063756,0.393211,12.815653,5.032583
5,PINN ($\lambda=0.01$),4.852055,0.385910,5.321299,0.407357,9.671036,5.557551
6,PINN ($\lambda=0.1$),6.021332,0.487565,6.126293,0.487615,1.743159,0.010232
7,PINN ($\lambda=0.5$),6.266083,0.494382,6.206056,0.493657,-0.957976,-0.146509
